# 11. Numbers

Why did the model write numbers badly, and did retraining on spoken-word numbers fix it?

In [1]:
from nbutil import *
import pandas as pd

## The casual-speech clip
One Enayet Chowdhury video. The released model writes the wrong digits; v1.1 writes the number in words.

In [2]:
pd.read_csv(ROOT / "results/numbers/enayet_clip_numbers.csv", encoding="utf-8-sig")

,time,v1.0 (released model),v1.1 (number fix),check
0,0:45,1900 সালে,উনিশশো চুরানব্বই সালে,said 1994 (Omi listened)
1,0:50,26 বছর,ছাব্বিশ বছর,consistent (1994 + 26 = 2020)
2,0:55,2200 সালে,দুই হাজার বিশ সালে,2020: Abraham Accords (UAE-Israel)
3,1:20,2220 সালে,দুই হাজার তেইশ সালে,year not verified
4,2:55,19222 সালে + 15x বিশ্ব loop,উনিশশো বাহাত্তর সালে,1972: Port Rashid opened
5,3:00,20000 সালে,দুই হাজার পাঁচ সালে,2005: DP World formed
6,3:10,10%,দশ পার্সেন্ট,~10% of world container traffic (DP World)
7,3:25,15 বছরের,পনের বছরের,same value
8,3:55,24%%%,চুয়াল্লিশ পারসেন্ট,not verified
9,4:35,35 টা দেশের 27 টা,পঁয়তাল্লিশটা দেশের সত্তরটা,not verified (the two versions disagree)


## How the training corpora write numbers

| Corpus | Clips with a digit |
|---|---|
| Common Voice, Kathbath, Shrutilipi | 0% |
| DL Sprint | 3.2% |
| OpenSLR 53 | 25.9% |

## The two training lists
Every training clip with digits was rewritten in spoken words. The reading (উনিশশো চুরানব্বই, চুরানব্বই, দুই হাজার বিশ, ...)
is the one the released model itself wrote with digit tokens blocked, kept only where it wrote every number of the
clip (3,473 of 6,188 clips); forced alignment could not hear 82% of the years, so it could not choose. v1.1 adds as
many number-word clips and other clips (73% of its hours ended up OpenSLR read news); v1.2 rebalances the filler to
the original source shares. Both: one epoch from the released model, learning rate 5e-6.

In [3]:
rows = {}
for name in ("mix_v1.1.json", "mix_v1.2.json"):
    m = load("results/numbers/" + name)
    rows[name] = {k: v for k, v in m.items() if not isinstance(v, dict)}
pd.DataFrame(rows).T

,report_clips,passed,rule,number_words,other,total,hours,judge_ok,mms_0.4_ok,both_ok,balanced
mix_v1.1.json,6387,3473,judge,3473,3473,10419,43.69,3473,2294,1559,NaN
mix_v1.2.json,6387,3473,judge,3322,7596,14391,45.29,3473,2294,1559,True


## Dev loss while training

In [4]:
a = pd.read_csv(ROOT / "results/numbers/v1.1/dev_log.csv")
b = pd.read_csv(ROOT / "results/numbers/v1.2/dev_log.csv")
a.insert(0, "run", "v1.1")
b.insert(0, "run", "v1.2")
pd.concat([a, b], ignore_index=True)

,run,step,dev_loss,dev_wer,dev_cer,n_utt,time
0,v1.1,0,0.2548,16.75,7.63,646,2026-11-03 22:20:41
1,v1.1,325,0.2697,17.45,8.08,646,2026-11-03 22:37:31
2,v1.2,0,0.2548,16.75,7.63,646,2026-11-03 23:27:50
3,v1.2,449,0.2677,17.29,7.92,646,2026-11-03 23:49:13


## Dev WER, split by whether the reference contains a digit

In [5]:
pd.read_csv(ROOT / "results/numbers/dev_split.csv", encoding="utf-8-sig").round(2)

,model,loops,clips_nodigit,wer_nodigit,clips_digit,wer_digit,clips_all,wer_all
0,v1.0,3,616,16.53,30,20.70,646,16.75
1,v1.1,4,616,17.07,30,24.18,646,17.45
2,v1.2,4,616,16.84,30,25.20,646,17.29


## Test clips with no digit in the reference
v1.2 minus v1.0, WER in percentage points.

In [6]:
t = pd.read_csv(ROOT / "results/numbers/test_drift.csv", encoding="utf-8-sig")
t["set"] = t["set"].map(SETS)
for c in ("wer_v10", "wer_v12", "diff", "ci_lo", "ci_hi"):
    t[c] = t[c].round(2)
t

,set,clips,wer_v10,wer_v12,diff,ci_lo,ci_hi,significant
0,DL Sprint eval,4072,11.00,11.30,0.30,0.13,0.47,True
1,FLEURS,733,13.60,13.60,0.00,-0.48,0.53,False
2,Common Voice,9408,10.14,10.38,0.24,0.12,0.37,True
3,Kathbath,2236,10.28,10.24,-0.05,-0.23,0.15,False


## Number value accuracy
Reference digit values that appear in the hypothesis, in any written form.

In [7]:
acc = pd.read_csv(ROOT / "results/numbers/number_accuracy.csv", encoding="utf-8-sig")
acc["set"] = acc["set"].map(SETS)
acc.assign(percent=(100 * acc["share"]).round(1))[["set", "model", "right", "total", "percent"]]

,set,model,right,total,percent
0,DL Sprint eval,v1.0,166,238,69.7
1,DL Sprint eval,v1.2,174,238,73.1
2,FLEURS,v1.0,163,293,55.6
3,FLEURS,v1.2,179,293,61.1


**Conclusion:** training text mixed digits and number words, and in digit mode the model guessed sound-alike values.
v1.1 and v1.2 write numbers in words and fix the casual-speech clip. On the test sets they get more numbers right
(DL Sprint 69.7% -> 73.1%, FLEURS 55.6% -> 61.1%) but drift on clips without digits (DL Sprint +0.30, Common Voice
+0.24, significant), so the released model stays v1.0 and the fix moves into the clean retrain.